# 01 — scanpy tutorial

My run of the scverse tutorial ["Preprocessing and clustering"](https://scanpy.scverse.org/en/stable/tutorials/basics/clustering.html): two human bone marrow samples, about 8,000 cells each. A learning step, kept as a record. Nothing is saved.

It walks the standard single-cell workflow end to end — QC, doublets, normalization, highly variable genes, PCA, neighbor graph, UMAP, Leiden clustering, then naming clusters from marker genes. `03_preprocess` applies the same steps to the Tabula Muris Senis Lung data, with the differences noted in each section below.

### 1. Setup

`scanpy` does the analysis, `anndata` holds the data, and `pooch` downloads the tutorial files and caches them so a rerun does not fetch them again. `set_figure_params` only affects how plots look.

In [ ]:
# Core scverse libraries
from __future__ import annotations

import anndata as ad

# Data retrieval
import pooch
import scanpy as sc

In [ ]:
sc.settings.set_figure_params(dpi=50, facecolor="white")

### 2. Load the two samples

Two 10x Genomics samples (`s1d1` and `s1d3`) published as `.h5` files, fetched by DOI. Two steps are worth noting:

- `var_names_make_unique()` because a 10x file can list the same gene symbol more than once, and duplicate names break later indexing.
- `ad.concat(..., label="sample")` stacks the samples into one object and records which sample each cell came from. That label is what makes per-sample QC and per-sample doublet detection possible further down. In `03_preprocess` the equivalent label is `mouse.id`, and it ends up mattering far more there: it is the unit the whole aging analysis holds out.

In [ ]:
EXAMPLE_DATA = pooch.create(
    path=pooch.os_cache("scverse_tutorials"),
    base_url="doi:10.6084/m9.figshare.22716739.v1/",
)
EXAMPLE_DATA.load_registry_from_doi()

In [ ]:
samples = {
    "s1d1": "s1d1_filtered_feature_bc_matrix.h5",
    "s1d3": "s1d3_filtered_feature_bc_matrix.h5",
}
adatas = {}

for sample_id, filename in samples.items():
    path = EXAMPLE_DATA.fetch(filename)
    sample_adata = sc.read_10x_h5(path)
    sample_adata.var_names_make_unique()
    adatas[sample_id] = sample_adata

adata = ad.concat(adatas, label="sample")
adata.obs_names_make_unique()
print(adata.obs["sample"].value_counts())
adata

### 3. Quality control

A gene group has to be flagged before it can be summarized, so mitochondrial, ribosomal, and hemoglobin genes are marked in `.var` first, then `calculate_qc_metrics` computes per-cell totals and the percentage coming from each group.

Why mitochondrial percentage is the key QC metric: when a cell's membrane breaks, its cytoplasmic mRNA leaks out, while mRNA inside the mitochondria stays trapped. A dying cell therefore reads as having an unusually high fraction of mitochondrial counts.

The violin and scatter plots come before the filter on purpose — the thresholds are chosen by looking at where the distributions actually sit, not from a remembered number. The filter here is deliberately loose (cells with fewer than 100 genes, genes in fewer than 3 cells), since clustering later exposes the cells that still need removing.

Two differences from the Lung data: the prefix is `MT-` in human and `mt-` in mouse, and the released Lung file contains no mitochondrial genes at all, so this check cannot be run there and is listed as a limitation instead.

In [ ]:
# mitochondrial genes, "MT-" for human, "Mt-" for mouse
adata.var["mt"] = adata.var_names.str.startswith("MT-")
# ribosomal genes
adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))
# hemoglobin genes
adata.var["hb"] = adata.var_names.str.contains("^HB[^(P)]")

In [ ]:
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt", "ribo", "hb"], inplace=True, log1p=True)

In [ ]:
sc.pl.violin(
    adata,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    jitter=0.4,
    multi_panel=True,
)

In [ ]:
sc.pl.scatter(adata, "total_counts", "n_genes_by_counts", color="pct_counts_mt")

In [ ]:
sc.pp.filter_cells(adata, min_genes=100)
sc.pp.filter_genes(adata, min_cells=3)

### 4. Doublets

A doublet is two cells caught in one droplet and sequenced under a single barcode, so it reads as one cell expressing two cell types' genes.

Scrublet builds artificial doublets by adding pairs of real cells together, then scores every real cell by how much it resembles those simulated ones. `batch_key="sample"` runs it per sample, because the doublet rate depends on how many cells were loaded into that particular run.

It must run **before** normalization: Scrublet normalizes internally, so feeding it already-normalized values distorts the simulation. In `03_preprocess` running it in the wrong order changed the call from 49 cells to 127, which is how I learned this mattered.

This step only flags cells, adding `predicted_doublet` and `doublet_score` to `.obs`.

In [ ]:
sc.pp.scrublet(adata, batch_key="sample")

### 5. Normalize and log-transform

The raw counts are copied into `layers["counts"]` first, because anything count-based later — differential expression, pseudobulk — needs integers, and both steps below overwrite `.X`.

Two steps, each fixing a different problem:

- **`normalize_total`** scales every cell to the same total. One cell may have 20,000 counts and another 5,000 purely because of capture efficiency, and without this that difference reads as the second cell expressing less of everything. Called with no `target_sum`, it scales to the median total across cells; `03_preprocess` uses `target_sum=1e4` instead, which is the same idea with a fixed number.
- **`log1p`** takes log(1 + x). Expression spans several orders of magnitude, and on the raw scale a handful of very high genes would dominate PCA; the log compresses that range. The `1 +` keeps zeros as zeros.

In [ ]:
# Saving count data
adata.layers["counts"] = adata.X.copy()

In [ ]:
# Normalizing to median total counts
sc.pp.normalize_total(adata)
# Logarithmize the data
sc.pp.log1p(adata)

### 6. Highly variable genes

Most genes are either off or uniformly expressed everywhere, and carry no information about which cell is which. This picks the 2,000 genes that vary most **relative to their mean**, since a highly expressed gene is noisier in absolute terms.

`batch_key="sample"` selects genes that are variable within each sample and then combines the rankings, so a gene that looks variable only because the two samples were processed differently does not get picked. Everything downstream — PCA, neighbors, clustering — uses these genes only.

In [ ]:
sc.pp.highly_variable_genes(adata, n_top_genes=2000, batch_key="sample")

In [ ]:
sc.pl.highly_variable_genes(adata)

### 7. PCA

PCA compresses the 2,000 genes into a smaller set of components, each a weighted combination of genes that vary together. Running clustering on the components rather than the raw genes removes a lot of noise and is much faster.

`pca_variance_ratio` shows how much variance each component carries; the curve flattens once the extra components are mostly noise, which is how the number used by `neighbors` is chosen.

The last plot is a check, not a result: coloring the first components by `sample` and by `pct_counts_mt` asks whether the dominant axes of variation are biology or something technical. A first component that tracks sample or mitochondrial content means a batch effect is driving the structure.

In [ ]:
sc.tl.pca(adata)

In [ ]:
sc.pl.pca_variance_ratio(adata, n_pcs=50, log=True)

In [ ]:
sc.pl.pca(
    adata,
    color=["sample", "sample", "pct_counts_mt", "pct_counts_mt"],
    dimensions=[(0, 1), (2, 3), (0, 1), (2, 3)],
    ncols=2,
    size=2,
)

### 8. Neighbor graph and UMAP

`neighbors` connects each cell to its nearest neighbors in PCA space, producing a graph where an edge means "these two cells have similar expression". That graph is what the clustering actually runs on.

`umap` lays the same graph out in two dimensions. It is **for looking at only**: distance on a UMAP is not quantitative, and the size and separation of blobs are artifacts of the layout rather than measurements. Coloring by `sample` checks that the two samples mix within each group instead of forming two parallel islands, which would mean a batch effect.

In [ ]:
sc.pp.neighbors(adata)

In [ ]:
sc.tl.umap(adata)

In [ ]:
sc.pl.umap(
    adata,
    color="sample",
    # Setting a smaller point size to get prevent overlap
    size=2,
)

### 9. Leiden clustering

Leiden finds communities in the neighbor graph: groups of cells more connected to each other than to the rest. `flavor="igraph"` with a fixed iteration count is the faster implementation that scanpy now recommends.

A cluster is a hypothesis about a cell type, not an answer. It gets a number, and naming it takes the marker genes in section 12.

In [ ]:
# Using the igraph implementation and a fixed number of iterations can be significantly faster,
# especially for larger datasets
sc.tl.leiden(adata, flavor="igraph", n_iterations=2)

In [ ]:
sc.pl.umap(adata, color=["leiden"])

### 10. Check QC metrics on the clusters

Clustering is also a QC tool. Plotting doublet score and the count metrics over the clusters asks whether any cluster is a technical artifact rather than a cell type — a cluster made mostly of predicted doublets, or one defined by low counts and high mitochondrial percentage, is a cluster of damaged cells.

This is why the earlier filter could stay loose: a problem that survives the thresholds usually becomes visible here as its own group.

In [ ]:
sc.pl.umap(
    adata,
    color=["leiden", "predicted_doublet", "doublet_score"],
    # increase horizontal space between panels
    wspace=0.5,
    size=3,
)

In [ ]:
sc.pl.umap(
    adata,
    color=["leiden", "log1p_total_counts", "pct_counts_mt", "log1p_n_genes_by_counts"],
    wspace=0.5,
    ncols=2,
)

### 11. Clustering at several resolutions

`resolution` controls how finely the graph is cut: 0.02 gives a few broad groups, 0.5 something intermediate, 2.0 many small ones.

There is no single correct value, and running several is the honest approach. The coarse resolution gives the big lineages, which are easy to name confidently; the fine one splits those into subtypes, some of which are real and some of which are noise. The usual workflow is to annotate at the coarse level first, then refine.

In [ ]:
for res in [0.02, 0.5, 2.0]:
    sc.tl.leiden(adata, key_added=f"leiden_res_{res:4.2f}", resolution=res, flavor="igraph")

In [ ]:
sc.pl.umap(
    adata,
    color=["leiden_res_0.02", "leiden_res_0.50", "leiden_res_2.00"],
    legend_loc="on data",
)

### 12. Name clusters with marker genes

`marker_genes` is a dictionary of known markers for each cell type expected in bone marrow, taken from the tutorial. The dotplot shows, for each cluster, how many cells express each gene (dot size) and how strongly (color).

`standard_scale="var"` scales each gene to 0–1 across clusters. Without it a highly expressed gene such as a hemoglobin gene would dominate the color scale and make everything else unreadable.

Reading it: find the cluster whose dots light up for one cell type's markers, and map that cluster number to the name. A few entries are **negative** markers, where the expected answer is absence — the comments in the dictionary flag these. The coarse clustering is named first (`cell_type_lvl1`), then the same plot is repeated at resolution 0.5 to see what each broad group split into.

In [ ]:
marker_genes = {
    "CD14+ Mono": ["FCN1", "CD14"],
    "CD16+ Mono": ["TCF7L2", "FCGR3A", "LYN"],
    # Note: DMXL2 should be negative
    "cDC2": ["CST3", "COTL1", "LYZ", "DMXL2", "CLEC10A", "FCER1A"],
    "Erythroblast": ["MKI67", "HBA1", "HBB"],
    # Note HBM and GYPA are negative markers
    "Proerythroblast": ["CDK6", "SYNGR1", "HBM", "GYPA"],
    "NK": ["GNLY", "NKG7", "CD247", "FCER1G", "TYROBP", "KLRG1", "FCGR3A"],
    "ILC": ["ID2", "PLCG2", "GNLY", "SYNE1"],
    "Naive CD20+ B": ["MS4A1", "IL4R", "IGHD", "FCRL1", "IGHM"],
    # Note IGHD and IGHM are negative markers
    "B cells": [
        "MS4A1",
        "ITGB1",
        "COL4A4",
        "PRDM1",
        "IRF4",
        "PAX5",
        "BCL11A",
        "BLK",
        "IGHD",
        "IGHM",
    ],
    "Plasma cells": ["MZB1", "HSP90B1", "FNDC3B", "PRDM1", "IGKC", "JCHAIN"],
    # Note PAX5 is a negative marker
    "Plasmablast": ["XBP1", "PRDM1", "PAX5"],
    "CD4+ T": ["CD4", "IL7R", "TRBC2"],
    "CD8+ T": ["CD8A", "CD8B", "GZMK", "GZMA", "CCL5", "GZMB", "GZMH", "GZMA"],
    "T naive": ["LEF1", "CCR7", "TCF7"],
    "pDC": ["GZMB", "IL3RA", "COBLL1", "TCF4"],
}

In [ ]:
sc.pl.dotplot(adata, marker_genes, groupby="leiden_res_0.02", standard_scale="var")

In [ ]:
adata.obs["cell_type_lvl1"] = adata.obs["leiden_res_0.02"].map(
    {
        "0": "Lymphocytes",
        "1": "Monocytes",
        "2": "Erythroid",
        "3": "B Cells",
    }
)

In [ ]:
sc.pl.dotplot(adata, marker_genes, groupby="leiden_res_0.50", standard_scale="var")

### 13. Genes that define each cluster

The reverse direction: instead of checking known markers, ask which genes are most enriched in each cluster. `rank_genes_groups` with a Wilcoxon test compares each cluster against all other cells, and the resulting top genes can be looked up to identify a cluster that the marker dictionary did not cover — here cluster 7, which turns out to be dendritic cells.

One important caveat, which carries into the aging analysis: this test treats every cell as an independent sample. Cells from the same sample are not independent, so the p-values come out far too small. The gene *ranking* is still useful for identifying a cluster; the p-values are not evidence of anything. Testing properly means aggregating to one value per sample first, which is what pseudobulk does.

In [ ]:
# Obtain cluster-specific differentially expressed genes
sc.tl.rank_genes_groups(adata, groupby="leiden_res_0.50", method="wilcoxon")

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata, groupby="leiden_res_0.50", standard_scale="var", n_genes=5)

In [ ]:
sc.get.rank_genes_groups_df(adata, group="7").head(5)

In [ ]:
dc_cluster_genes = sc.get.rank_genes_groups_df(adata, group="7").head(5)["names"]
sc.pl.umap(
    adata,
    color=[*dc_cluster_genes, "leiden_res_0.50"],
    legend_loc="on data",
    frameon=False,
    ncols=3,
)

### What carried over to the Lung analysis

- **Order matters.** Scrublet on raw counts, before normalization. Save the counts layer before overwriting `.X`.
- **Per-sample everything.** QC, doublet detection, and highly variable gene selection all take the sample label. In the Lung data the sample is the mouse, and it becomes the unit of analysis for every model.
- **Plots before thresholds.** Every filter here was chosen after looking at the distribution.
- **Clustering is a check on annotation, not a replacement.** The Lung data arrives already annotated by the consortium, so there the Leiden clusters are used to verify those labels and to spot one mouse whose cells sit apart from everyone else's.
- **A per-cell test overstates significance.** The last section's caveat is the reason the aging analysis averages per mouse before comparing groups.

Two things here cannot be reused on the Lung data: there are no mitochondrial genes in that release, so the main QC metric is unavailable, and the gene symbols are the older mouse names, so a marker list has to be checked against `adata.var_names` rather than assumed.
